# MerchantHybridEncoder 데모

`KLUE-RoBERTa + UTF-8 Byte Transformer + Gated Fusion + 계층형 분류` 구조를 샘플 데이터로 학습하고 테스트합니다.

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 권장합니다.

## 1. ZIP 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_files = [name for name in uploaded if name.endswith('.zip')]

if not zip_files:
    raise ValueError('merchant_hybrid_colab_demo.zip을 업로드해 주세요.')

zip_name = zip_files[0]
print('업로드:', zip_name)

## 2. 압축 해제

In [ ]:
import os
import shutil
import zipfile

project_dir = '/content/merchant_hybrid_colab_demo'

if os.path.exists(project_dir):
    shutil.rmtree(project_dir)

with zipfile.ZipFile(zip_name, 'r') as archive:
    archive.extractall('/content')

os.chdir(project_dir)
print('작업 폴더:', os.getcwd())
print('\n'.join(sorted(os.listdir('.'))))

## 3. 라이브러리 설치

In [ ]:
!pip -q install -r requirements.txt

## 4. 실행 장치 확인

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 5. 전체 Fine-tuning 데모

처음 실행할 때 `klue/roberta-base` 모델이 다운로드됩니다.

In [ ]:
!python main.py --mode demo --device auto --epochs 3 --batch-size 8

## 6. 결과 확인

In [ ]:
import pandas as pd

test_result = pd.read_csv('result/test_result.csv')
inference_result = pd.read_csv('result/inference_result.csv')

print('테스트 결과')
display(test_result)

print('추론 결과')
display(inference_result)

## 7. 원하는 문장으로 추론

In [ ]:
custom_texts = [
    '메가MGC커피 강남역2호점',
    '서울뚝배기 김치찌개 백반',
    'GS25동탄센트럴24H',
    '행복온누리약국 처방전조제',
    'URBAN FASHION 여성 캐주얼',
]

pd.DataFrame({
    'id': range(len(custom_texts)),
    'text': custom_texts,
}).to_csv('data/pred_data.csv', index=False)

!python main.py --mode inference --device auto

display(pd.read_csv('result/inference_result.csv'))

## 8. 빠른 Smoke Test 대안

런타임이 부족하면 새 런타임에서 아래 명령으로 Text Encoder를 고정한 연결 테스트를 실행할 수 있습니다.

In [ ]:
# 필요할 때 주석을 제거하세요.
# !python main.py --mode demo --device auto --epochs 2 --batch-size 8 --freeze-text-encoder

## 9. 결과 다운로드

In [ ]:
from google.colab import files

files.download('result/inference_result.csv')